# DeepFilterNet faithful 16 kHz cho Cortex-M55 / Ethos-U55

Notebook Kaggle self-contained này train DeepFilterNet đơn kênh trên dataset
`speech_enhancement_v1`. Kiến trúc giữ hai-stage gốc: ERB mask, deep complex filter
và alpha blend. Mọi state streaming là tensor I/O explicit để export TFLite INT8/Vela.

## Chạy đúng thứ tự

1. Upload package Kaggle tạo từ `Tuần 3/tools/create_kaggle_exports.py --apply`.
2. Chạy `RUN_PROFILE = "debug"` trước; profile này chạy train/export/Vela/golden mini.
3. Chỉ khi debug pass, đổi sang `RUN_PROFILE = "full"` để train đầy đủ.
4. Final test chỉ chạy checkpoint frozen, không train cùng phiên.


## Môi trường Kaggle và profile chạy

**Mục tiêu:** Cài dependency tối thiểu, khóa profile debug/full/final và in log cấu hình.

**Đầu vào:** Kaggle Python image, Ethos-U Vela, TensorFlow và profile mong muốn.

**Đầu ra:** RUN_CONFIG đã validate; debug chạy toàn pipeline nhỏ trước full train.

**Quyết định kỹ thuật:** Không nâng TensorFlow của Kaggle. Debug dùng PTQ để bắt API/export lỗi; full dùng constrained fine-tune rồi calibrate full INT8.

**Lỗi thường gặp:** Chạy cell không tuần tự, sửa shape/state contract hoặc dùng sai dataset manifest. Dừng ở assertion đầu tiên; không bỏ quality gate để tiếp tục export.

**Artifact sinh ra:** RUN_CONFIG đã validate; debug chạy toàn pipeline nhỏ trước full train.


In [1]:
import importlib.metadata
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf

for package in ("ethos-u-vela", "pesq", "pystoi"):
    try:
        importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package])

from pesq import pesq as pesq_score
from pystoi import stoi as stoi_score

MODEL_NAME = "deepfilternet_u55"
RUN_PROFILE = "full"  # debug -> full -> final -> merge
EVALUATION_MODE = "auto"  # auto -> profile default; final may be shard, merge reads shards only
EVAL_SHARD_INDEX = 0
EVAL_SHARD_SIZE = 250
EVAL_PARTIAL_EVERY = 16
FROZEN_CHECKPOINT_MANIFEST_SHA256 = None
FROZEN_CHECKPOINT_ROOT = Path("/kaggle/input/deepfilternet-u55-checkpoint")
FROZEN_PACKAGE_ROOT = Path("/kaggle/input/deepfilternet-u55-package/integration_package")
EVAL_SHARD_INPUT_ROOT = Path("/kaggle/input/deepfilternet-u55-eval-shards")
SEED = 20260805
DATASET_ROOT_HINT = Path("/kaggle/input/speech-enhancement-v1")
VELA_ACCELERATOR = "ethos-u55-256"
MODEL_VARIANT = "faithful"
RUN_DIR = Path("/kaggle/working") / f"{MODEL_NAME}_{RUN_PROFILE}_seed{SEED}"

PROFILE = {
    "debug": dict(train=True, export=True, validate=True, final=False, batch=1,
                  fp32_max_epochs=1, qat_max_epochs=1,
                  fp32_steps_per_virtual_epoch=2, qat_steps_per_virtual_epoch=2,
                  qat_calibration_batches=1, qat_crop_samples=4096,
                  validation_every=1, val_items=2, metric_items=2, evaluation_mode="smoke"),
    "full": dict(train=True, export=True, validate=True, final=False, batch=8,
                 fp32_max_epochs=22, qat_max_epochs=4,
                 fp32_steps_per_virtual_epoch=400, qat_steps_per_virtual_epoch=200,
                 qat_calibration_batches=8, qat_crop_samples=8192,
                 validation_every=2, val_items=256, metric_items=512, evaluation_mode="smoke"),
    "final": dict(train=False, export=False, validate=False, final=True, batch=1,
                  fp32_max_epochs=0, qat_max_epochs=0,
                  fp32_steps_per_virtual_epoch=0, qat_steps_per_virtual_epoch=0,
                  qat_calibration_batches=0, qat_crop_samples=32000,
                  validation_every=1, val_items=0, metric_items=None, evaluation_mode="shard"),
    "merge": dict(train=False, export=False, validate=False, final=False, batch=1,
                  fp32_max_epochs=0, qat_max_epochs=0,
                  fp32_steps_per_virtual_epoch=0, qat_steps_per_virtual_epoch=0,
                  qat_calibration_batches=0, qat_crop_samples=32000,
                  validation_every=1, val_items=0, metric_items=None, evaluation_mode="merge"),
}
FP32_EARLY_STOP_START, FP32_EARLY_STOP_PATIENCE = 6, 4
QAT_EARLY_STOP_START, QAT_EARLY_STOP_PATIENCE = 3, 3
FP32_MAX_EPOCHS, QAT_MAX_EPOCHS = PROFILE["full"]["fp32_max_epochs"], PROFILE["full"]["qat_max_epochs"]
FLASH_RELEASE_BUDGET_BYTES = int(1.8 * 1024 * 1024)
QUALITY_LIMITS = {"delta_si_sdr_db": 0.5, "delta_pesq": 0.05, "delta_stoi": 0.01}
if RUN_PROFILE not in PROFILE:
    raise ValueError(f"RUN_PROFILE phải là một trong {sorted(PROFILE)}")
RUN_CONFIG = PROFILE[RUN_PROFILE]
if EVALUATION_MODE == "auto":
    EVALUATION_MODE = RUN_CONFIG["evaluation_mode"]
if EVALUATION_MODE not in {"smoke", "shard", "merge"}:
    raise ValueError("EVALUATION_MODE phải là smoke, shard hoặc merge")
if RUN_PROFILE == "full" and EVALUATION_MODE != "smoke":
    raise ValueError("full chỉ chạy train/export/smoke; full test dùng final shard")
if RUN_PROFILE == "final" and EVALUATION_MODE != "shard":
    raise ValueError("final chỉ chạy một shard; merge dùng RUN_PROFILE='merge'")
if RUN_PROFILE == "merge" and EVALUATION_MODE != "merge":
    raise ValueError("merge không được infer model")
if RUN_CONFIG["final"] and not (FROZEN_CHECKPOINT_MANIFEST_SHA256 and len(FROZEN_CHECKPOINT_MANIFEST_SHA256) == 64):
    raise ValueError("Profile final yêu cầu FROZEN_CHECKPOINT_MANIFEST_SHA256")
if EVAL_SHARD_SIZE != 250 or EVAL_PARTIAL_EVERY != 16:
    raise ValueError("Release contract khóa shard=250 và partial checkpoint=16")

tf.keras.utils.set_random_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print({"model": MODEL_NAME, "profile": RUN_PROFILE, "evaluation_mode": EVALUATION_MODE,
       "config": RUN_CONFIG, "tensorflow": tf.__version__})


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.8 MB/s eta 0:00:00
{'model': 'deepfilternet_u55', 'profile': 'full', 'evaluation_mode': 'smoke', 'config': {'train': True, 'export': True, 'validate': True, 'final': False, 'batch': 8, 'fp32_max_epochs': 22, 'qat_max_epochs': 4, 'fp32_steps_per_virtual_epoch': 400, 'qat_steps_per_virtual_epoch': 200, 'qat_calibration_batches': 8, 'qat_crop_samples': 8192, 'validation_every': 2, 'val_items': 256, 'metric_items': 512, 'evaluation_mode': 'smoke'}, 'tensorflow': '2.20.0'}


## Dataset manifest speech_enhancement_v1

**Mục tiêu:** Tìm package Kaggle theo manifest direct-pair như notebook template, kiểm split và nạp audio PCM16 mono 16 kHz.

**Đầu vào:** `manifests/pairs_{train,val,test}.csv` với mixture_relative_path và clean_relative_path.

**Đầu ra:** DataFrame split, tf.data dataset crop đồng bộ 2 giây và run metadata.

**Quyết định kỹ thuật:** Không phụ thuộc Kaggle slug; test split chỉ được dùng ở profile final.

**Lỗi thường gặp:** Chạy cell không tuần tự, sửa shape/state contract hoặc dùng sai dataset manifest. Dừng ở assertion đầu tiên; không bỏ quality gate để tiếp tục export.

**Artifact sinh ra:** DataFrame split, tf.data dataset crop đồng bộ 2 giây và run metadata.


In [2]:
PAIR_COLUMNS = ("mixture_relative_path", "clean_relative_path")
SAMPLE_RATE = 16000
CROP_SAMPLES = SAMPLE_RATE * 2

def resolve_dataset_root(root_hint):
    candidates = [Path(root_hint), Path(root_hint) / "speech_enhancement_v1"]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        candidates += [path for path in input_root.iterdir() if path.is_dir()]
        candidates += [path.parent.parent for path in input_root.rglob("pairs_train.csv") if path.parent.name == "manifests"]
    for root in dict.fromkeys(candidates):
        if all((root / "manifests" / f"pairs_{split}.csv").is_file() for split in ("train", "val", "test")):
            return root
    raise FileNotFoundError("Không tìm thấy speech_enhancement_v1 có đủ pairs_train/val/test.csv")

ROOT = resolve_dataset_root(DATASET_ROOT_HINT)

def load_pairs(split):
    frame = pd.read_csv(ROOT / "manifests" / f"pairs_{split}.csv")
    missing = set(PAIR_COLUMNS) - set(frame.columns)
    normalized = frame.loc[:, PAIR_COLUMNS].dropna().astype(str).reset_index(drop=True) if not missing else frame
    if missing or normalized.empty:
        raise ValueError(f"pairs_{split}.csv không hợp lệ sau normalize: missing={sorted(missing)} rows={len(normalized)} raw_rows={len(frame)}")
    return normalized

train_pairs, val_pairs, test_pairs = (load_pairs(split) for split in ("train", "val", "test"))
owners = {}
for split, frame in (("train", train_pairs), ("val", val_pairs), ("test", test_pairs)):
    for relative in frame.to_numpy().flat:
        key = relative.replace("\\", "/").lower()
        if key in owners and owners[key] != split:
            raise ValueError(f"Audio leak giữa {owners[key]} và {split}: {relative}")
        owners[key] = split

def read_wav_16k_mono(path):
    audio, rate = tf.audio.decode_wav(tf.io.read_file(path), desired_channels=1)
    tf.debugging.assert_equal(rate, SAMPLE_RATE, message="Dataset phải là 16 kHz")
    return tf.squeeze(audio, axis=-1)

def make_dataset(frame, training, limit=None, crop_samples=None, batch_size=None):
    frame = frame.iloc[:limit].copy() if limit is not None else frame.copy()
    crop_samples = CROP_SAMPLES if crop_samples is None else int(crop_samples)
    batch_size = RUN_CONFIG["batch"] if batch_size is None else int(batch_size)

    if frame.empty:
        split_name = "train" if training else "validation/test"
        raise ValueError(
            f"Dataset split rỗng: split={split_name}, limit={limit}. "
            "Chạy lại cell manifest và kiểm tra số pair hợp lệ."
        )
    if batch_size < 1:
        raise ValueError(f"batch_size phải dương, nhận {batch_size}")
    if crop_samples < FRAME_LENGTH:
        raise ValueError(f"crop_samples phải >= {FRAME_LENGTH}, nhận {crop_samples}")

    paths = np.asarray(
        [(ROOT / value).as_posix() for value in frame[PAIR_COLUMNS[0]]],
        dtype=str,
    )
    clean = np.asarray(
        [(ROOT / value).as_posix() for value in frame[PAIR_COLUMNS[1]]],
        dtype=str,
    )
    dataset = tf.data.Dataset.from_tensor_slices((paths, clean))

    if training:
        dataset = dataset.shuffle(
            min(len(frame), 8192), seed=SEED, reshuffle_each_iteration=True
        ).repeat()

    def decode(mix_path, clean_path):
        mixture = read_wav_16k_mono(mix_path)
        target = read_wav_16k_mono(clean_path)
        length = tf.minimum(tf.shape(mixture)[0], tf.shape(target)[0])
        tf.debugging.assert_greater_equal(length, crop_samples)

        start = (
            tf.random.uniform([], 0, length - crop_samples + 1, tf.int32, seed=SEED)
            if training else (length - crop_samples) // 2
        )
        return (
            mixture[start:start + crop_samples],
            target[start:start + crop_samples],
        )

    return (
        dataset.map(
            decode,
            num_parallel_calls=tf.data.AUTOTUNE,
            deterministic=not training,
        )
        .batch(batch_size, drop_remainder=training)
        .prefetch(tf.data.AUTOTUNE)
    )

RUN_METADATA = {"model": MODEL_NAME, "profile": RUN_PROFILE, "dataset_root": str(ROOT), "counts": {"train": len(train_pairs), "val": len(val_pairs), "test": len(test_pairs)}}
(RUN_DIR / "run_metadata.json").write_text(json.dumps(RUN_METADATA, indent=2), encoding="utf-8")
print(RUN_METADATA)


{'model': 'deepfilternet_u55', 'profile': 'full', 'dataset_root': '/kaggle/input/datasets/minhhuybi/speech-enhancement-v1/speech_enhancement_v1', 'counts': {'train': 17500, 'val': 3750, 'test': 3750}}


## DSP contract 512/256, 32 ERB và 144 deep-filter bins

**Mục tiêu:** Khóa frontend faithful 16 kHz, ERB bank, fixed inverse-ERB và kiểm reconstruction.

**Đầu vào:** Waveform 16 kHz và các hằng số kiến trúc Wögerbauer.

**Đầu ra:** STFT `[B,T,257]`, ERB `[B,T,32]`, complex `[B,T,144,2]`.

**Quyết định kỹ thuật:** Inverse-ERB nằm trong graph dưới dạng ma trận hằng để firmware không phải lặp expansion 32→257.

**Lỗi thường gặp:** Chạy cell không tuần tự, sửa shape/state contract hoặc dùng sai dataset manifest. Dừng ở assertion đầu tiên; không bỏ quality gate để tiếp tục export.

**Artifact sinh ra:** STFT `[B,T,257]`, ERB `[B,T,32]`, complex `[B,T,144,2]`.


In [3]:
FRAME_LENGTH = 512
FRAME_STEP = 256
FFT_BINS = 257
ERB_BANDS = 32
DF_BINS = 144
DF_TAPS = 5
COMPLEX_FEATURE_SIZE = DF_BINS * 2

def sqrt_hann(length, dtype=tf.float32):
    return tf.sqrt(tf.signal.hann_window(length, periodic=True, dtype=dtype))

def stft(audio):
    return tf.signal.stft(audio, FRAME_LENGTH, FRAME_STEP, FRAME_LENGTH, window_fn=sqrt_hann, pad_end=False)

def istft(spectrum):
    return tf.signal.inverse_stft(spectrum, FRAME_LENGTH, FRAME_STEP, FRAME_LENGTH, window_fn=tf.signal.inverse_stft_window_fn(FRAME_STEP, forward_window_fn=sqrt_hann))

def erb_bank():
    frequencies = np.arange(FFT_BINS, dtype=np.float32) * SAMPLE_RATE / FRAME_LENGTH
    erb = 21.4 * np.log10(1.0 + 0.00437 * frequencies)
    centers = np.linspace(erb[0], erb[-1], ERB_BANDS, dtype=np.float32)
    analysis = np.maximum(1.0 - np.abs(erb[:, None] - centers[None, :]) / max(centers[1] - centers[0], 1e-6), 0.0)
    analysis /= np.maximum(analysis.sum(axis=0, keepdims=True), 1e-8)
    expansion = analysis.T.copy()
    expansion /= np.maximum(expansion.sum(axis=0, keepdims=True), 1e-8)
    return analysis.astype(np.float32), expansion.astype(np.float32)

ERB_ANALYSIS_NP, ERB_EXPANSION_NP = erb_bank()
ERB_ANALYSIS = tf.constant(ERB_ANALYSIS_NP)
ERB_EXPANSION = tf.constant(ERB_EXPANSION_NP)

def make_features(spectrum):
    magnitude = tf.abs(spectrum)
    erb_feature = tf.math.log(tf.maximum(tf.linalg.matmul(magnitude, ERB_ANALYSIS), 1e-6))
    low = spectrum[..., :DF_BINS]
    complex_feature = tf.stack([tf.math.real(low), tf.math.imag(low)], axis=-1)
    return erb_feature, complex_feature

probe = tf.random.uniform([1, 8192], -0.5, 0.5, seed=SEED)
rebuilt = istft(stft(probe))
margin = FRAME_LENGTH
reconstruction_error = float(tf.reduce_max(tf.abs(probe[:, margin:-margin] - rebuilt[:, margin:-margin])))
if reconstruction_error >= 1e-5:
    raise RuntimeError(f"STFT/iSTFT mismatch: {reconstruction_error}")
print({"frontend": [FRAME_LENGTH, FRAME_STEP, FFT_BINS], "erb": ERB_ANALYSIS.shape, "df": [DF_TAPS, DF_BINS, 2], "reconstruction_error": reconstruction_error})


I0000 00:00:1785860746.012018      23 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


{'frontend': [512, 256, 257], 'erb': TensorShape([257, 32]), 'df': [5, 144, 2], 'reconstruction_error': 2.384185791015625e-07}


## DeepFilterNet frame core: grouped-GRU equivalent, decoder và state contract

**Mục tiêu:** Tạo frame core causal có state I/O explicit; grouped linear/GRU được materialize bằng Dense primitive theo từng group.

**Đầu vào:** ERB frame, complex frame, 5 recurrent state và hai causal convolution caches.

**Đầu ra:** Bin gains `[1,257]`, filter coefficients `[1,5,144,2]`, alpha `[1,1]` và next-state tensors.

**Quyết định kỹ thuật:** Không dùng BatchNorm hay tf.keras.GRU. Group math giữ nguyên nhưng export thành FullyConnected/Add/Mul/Sigmoid/Tanh.

**Lỗi thường gặp:** Chạy cell không tuần tự, sửa shape/state contract hoặc dùng sai dataset manifest. Dừng ở assertion đầu tiên; không bỏ quality gate để tiếp tục export.

**Artifact sinh ra:** Bin gains `[1,257]`, filter coefficients `[1,5,144,2]`, alpha `[1,1]` và next-state tensors.


In [4]:
MODEL_VARIANT = globals().get("MODEL_VARIANT", "faithful")
if MODEL_VARIANT == "faithful":
    ENC_GROUPS, ENC_UNITS, DF_GROUPS, DF_UNITS = 4, 80, 8, 40
elif MODEL_VARIANT == "compact":
    ENC_GROUPS, ENC_UNITS, DF_GROUPS, DF_UNITS = 4, 64, 8, 32
else:
    raise ValueError("MODEL_VARIANT phải là faithful hoặc compact")
ENC_STATE_SIZE, DF_STATE_SIZE = ENC_GROUPS * ENC_UNITS, DF_GROUPS * DF_UNITS
STATE_NAMES = tuple([f"enc_state_{index}" for index in range(3)] + [f"df_state_{index}" for index in range(2)] + ["erb_conv_cache", "complex_conv_cache"])

class FakeQuantizer(tf.keras.layers.Layer):
    """Train-only INT8 fake quant; disabled before SavedModel/TFLite export."""
    def __init__(self, per_channel=False, name=None):
        super().__init__(name=name, dtype="float32", autocast=False)
        self.per_channel, self.enabled = per_channel, False
        self.observe, self.sample_saturation = False, False
    def build(self, shape):
        range_shape = (int(shape[-1]),) if self.per_channel else ()
        self.minimum = self.add_weight(name="minimum", shape=range_shape, initializer=tf.keras.initializers.Constant(-1.0), trainable=False)
        self.maximum = self.add_weight(name="maximum", shape=range_shape, initializer=tf.keras.initializers.Constant(1.0), trainable=False)
        self.observed = self.add_weight(name="observed", shape=(), dtype=tf.int64, initializer="zeros", trainable=False)
        self.saturated = self.add_weight(name="saturated", shape=(), dtype=tf.int64, initializer="zeros", trainable=False)
    def call(self, values, training=False):
        if not self.enabled:
            return values
        if self.observe:
            axes = tuple(range(values.shape.rank - 1)) if self.per_channel else None
            peak = tf.maximum(tf.reduce_max(tf.abs(values), axis=axes), tf.constant(1e-6, tf.float32))
            self.minimum.assign(tf.minimum(self.minimum, -peak))
            self.maximum.assign(tf.maximum(self.maximum, peak))
        if self.sample_saturation:
            below, above = values < self.minimum, values > self.maximum
            self.observed.assign_add(tf.cast(tf.size(values), tf.int64))
            self.saturated.assign_add(tf.cast(tf.math.count_nonzero(tf.logical_or(below, above)), tf.int64))
        if self.per_channel:
            return tf.quantization.fake_quant_with_min_max_vars_per_channel(values, self.minimum, self.maximum, num_bits=8, narrow_range=True)
        return tf.quantization.fake_quant_with_min_max_vars(values, self.minimum, self.maximum, num_bits=8, narrow_range=True)

class QATDense(tf.keras.layers.Dense):
    def __init__(self, units, **kwargs):
        kwargs.setdefault("autocast", False)
        super().__init__(units, **kwargs)
        self.kernel_quantizer = FakeQuantizer(
            per_channel=True, name=f"{self.name}_kernel_q"
        )
        self.output_quantizer = FakeQuantizer(name=f"{self.name}_output_q")

    def call(self, inputs, training=False):
        kernel = self.kernel_quantizer(self.kernel, training=training)
        outputs = tf.linalg.matmul(inputs, kernel)
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return self.output_quantizer(outputs, training=training)

class QATConv2D(tf.keras.layers.Conv2D):
    def __init__(self, filters, kernel_size, **kwargs):
        kwargs.setdefault("autocast", False)
        super().__init__(filters, kernel_size, **kwargs)
        self.kernel_quantizer = FakeQuantizer(
            per_channel=True, name=f"{self.name}_kernel_q"
        )
        self.output_quantizer = FakeQuantizer(name=f"{self.name}_output_q")

    def call(self, inputs, training=False):
        kernel = self.kernel_quantizer(self.kernel, training=training)
        outputs = tf.nn.conv2d(
            inputs,
            kernel,
            strides=[1, *self.strides, 1],
            padding=self.padding.upper(),
            dilations=[1, *self.dilation_rate, 1],
        )
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return self.output_quantizer(outputs, training=training)

def set_qat_enabled(model, enabled):
    layers = model._flatten_layers(include_self=True, recursive=True)
    for layer in layers:
        if isinstance(layer, FakeQuantizer):
            layer.enabled = bool(enabled)
    print({"qat_enabled": bool(enabled), "fake_quantizers": sum(isinstance(layer, FakeQuantizer) for layer in layers), "observer": False})

def set_qat_observer(model, enabled):
    for layer in model._flatten_layers(include_self=True, recursive=True):
        if isinstance(layer, FakeQuantizer):
            layer.observe = bool(enabled)
    print({"qat_observer": bool(enabled)})

def refresh_kernel_ranges(model):
    refreshed = 0
    for layer in model._flatten_layers(include_self=True, recursive=True):
        if isinstance(layer, (QATDense, QATConv2D)):
            quantizer, kernel = layer.kernel_quantizer, layer.kernel
            axes = tuple(range(kernel.shape.rank - 1))
            peak = tf.maximum(tf.reduce_max(tf.abs(kernel), axis=axes), tf.constant(1e-6, tf.float32))
            quantizer.minimum.assign(-peak); quantizer.maximum.assign(peak); refreshed += 1
    print({"qat_kernel_ranges_refreshed": refreshed})

def set_qat_saturation_sampling(model, enabled):
    for layer in model._flatten_layers(include_self=True, recursive=True):
        if isinstance(layer, FakeQuantizer):
            layer.sample_saturation = bool(enabled)
            if enabled:
                layer.observed.assign(0); layer.saturated.assign(0)

def qat_range_report(model):
    report = {}
    for layer in model._flatten_layers(include_self=True, recursive=True):
        if isinstance(layer, FakeQuantizer):
            observed = int(layer.observed.numpy())
            report[layer.name] = {"minimum": np.asarray(layer.minimum.numpy()).tolist(), "maximum": np.asarray(layer.maximum.numpy()).tolist(), "observed": observed, "saturated": int(layer.saturated.numpy()), "saturation_ratio": float(layer.saturated.numpy() / max(observed, 1))}
    return report

class GroupedGRU(tf.keras.layers.Layer):
    def __init__(self, groups, units, name):
        super().__init__(name=name, dtype="float32", autocast=False)
        self.groups, self.units = groups, units
        self.input_dense = [QATDense(3 * units, dtype="float32", name=f"{name}_input_{i}") for i in range(groups)]
        self.state_dense = [QATDense(3 * units, use_bias=False, dtype="float32", name=f"{name}_state_{i}") for i in range(groups)]
        self.state_quantizer = FakeQuantizer(name=f"{name}_state_q")
    def call(self, x, state, training=False):
        x_parts, state_parts, next_parts = tf.split(x, self.groups, axis=-1), tf.split(state, self.groups, axis=-1), []
        for x_part, state_part, input_dense, state_dense in zip(x_parts, state_parts, self.input_dense, self.state_dense):
            gates = input_dense(x_part, training=training) + state_dense(state_part, training=training)
            update, reset, candidate = tf.split(gates, 3, axis=-1)
            update, reset = tf.sigmoid(update), tf.sigmoid(reset)
            candidate = tf.tanh(candidate + reset * state_part)
            next_parts.append(update * state_part + (1.0 - update) * candidate)
        return self.state_quantizer(tf.concat(next_parts, axis=-1), training=training)

class DeepFilterNetCell(tf.keras.layers.Layer):
    def __init__(self):
        super().__init__(name="deepfilternet_u55_cell", dtype="float32", autocast=False)
        self.erb_conv1 = QATConv2D(18, (2, 3), padding="valid", activation="relu", dtype="float32")
        self.erb_conv2 = QATConv2D(36, (1, 3), strides=(1, 2), padding="same", activation="relu", dtype="float32")
        self.erb_conv3 = QATConv2D(36, (1, 3), strides=(1, 2), padding="same", activation="relu", dtype="float32")
        self.erb_skip = QATConv2D(36, (1, 1), padding="same", dtype="float32")
        self.complex_conv1 = QATConv2D(18, (2, 3), padding="valid", activation="relu", dtype="float32")
        self.complex_conv2 = QATConv2D(36, (1, 3), strides=(1, 2), padding="same", activation="relu", dtype="float32")
        self.complex_reduce = tf.keras.layers.AveragePooling2D((1, 9), strides=(1, 9), dtype="float32", autocast=False)
        self.erb_group_linear = [QATDense(40, activation="relu", dtype="float32", name=f"erb_group_linear_{i}") for i in range(8)]
        self.complex_group_linear = [QATDense(40, activation="relu", dtype="float32", name=f"complex_group_linear_{i}") for i in range(8)]
        self.fuse = QATDense(ENC_STATE_SIZE, activation="relu", dtype="float32")
        self.enc_grus = [GroupedGRU(ENC_GROUPS, ENC_UNITS, f"enc_grouped_gru_{i}") for i in range(3)]
        self.erb_decoder = QATDense(ERB_BANDS, activation="sigmoid", dtype="float32", name="erb_decoder")
        self.alpha_decoder = QATDense(1, activation="sigmoid", dtype="float32", name="alpha_decoder")
        self.df_grus = [GroupedGRU(DF_GROUPS, DF_UNITS, f"df_grouped_gru_{i}") for i in range(2)]
        self.df_heads = [QATDense(DF_TAPS * (DF_BINS // DF_GROUPS) * 2, activation="tanh", dtype="float32", name=f"df_group_linear_{i}") for i in range(DF_GROUPS)]
    def call(self, erb_frame, complex_frame, states, training=False):
        erb_cache, complex_cache = states[-2:]
        erb_input = tf.concat([erb_cache[:, None, :, None], erb_frame[:, None, :, None]], axis=1)
        erb_input = tf.pad(erb_input, [[0, 0], [0, 0], [1, 1], [0, 0]])
        complex_current = complex_frame[:, None, :, :]
        complex_input = tf.concat([complex_cache, complex_current], axis=1)
        complex_input = tf.pad(complex_input, [[0, 0], [0, 0], [1, 1], [0, 0]])
        erb_x = self.erb_conv3(self.erb_conv2(self.erb_conv1(erb_input, training=training), training=training), training=training)
        erb_x = erb_x + self.erb_skip(erb_x, training=training)
        complex_x = self.complex_reduce(self.complex_conv2(self.complex_conv1(complex_input, training=training), training=training), training=training)
        erb_x = tf.reshape(erb_x, [-1, 8, 36])
        complex_x = tf.reshape(complex_x, [-1, 8, 36])
        erb_x = tf.concat([layer(erb_x[:, index], training=training) for index, layer in enumerate(self.erb_group_linear)], axis=-1)
        complex_x = tf.concat([layer(complex_x[:, index], training=training) for index, layer in enumerate(self.complex_group_linear)], axis=-1)
        latent = self.fuse(tf.concat([erb_x, complex_x], axis=-1), training=training)
        next_states = []
        for layer, state in zip(self.enc_grus, states[:3]):
            latent = layer(latent, state, training=training)
            next_states.append(latent)
        erb_gain = self.erb_decoder(latent, training=training)
        alpha = self.alpha_decoder(latent, training=training)
        df_latent = latent
        for layer, state in zip(self.df_grus, states[3:5]):
            df_latent = layer(df_latent, state, training=training)
            next_states.append(df_latent)
        coefficients = tf.concat([head(part, training=training) for head, part in zip(self.df_heads, tf.split(df_latent, DF_GROUPS, axis=-1))], axis=-1)
        coefficients = tf.reshape(coefficients, [-1, DF_TAPS, DF_BINS, 2])
        gain_bins = tf.linalg.matmul(erb_gain, ERB_EXPANSION)
        next_states += [erb_frame, complex_current]
        return gain_bins, coefficients, alpha, next_states

class DeepFilterNetTrain(tf.keras.Model):
    def __init__(self):
        super().__init__(name=MODEL_NAME, dtype="float32", autocast=False)
        self.cell = DeepFilterNetCell()
    def call(self, erb_sequence, complex_sequence, training=False):
        batch, frames = tf.shape(erb_sequence)[0], tf.shape(erb_sequence)[1]
        states = [tf.zeros([batch, ENC_STATE_SIZE], tf.float32) for _ in range(3)] + [tf.zeros([batch, DF_STATE_SIZE], tf.float32) for _ in range(2)] + [tf.zeros([batch, ERB_BANDS], tf.float32), tf.zeros([batch, 1, DF_BINS, 2], tf.float32)]
        gains = tf.TensorArray(tf.float32, size=frames, element_shape=tf.TensorShape([None, FFT_BINS]))
        coefficients = tf.TensorArray(tf.float32, size=frames, element_shape=tf.TensorShape([None, DF_TAPS, DF_BINS, 2]))
        alphas = tf.TensorArray(tf.float32, size=frames, element_shape=tf.TensorShape([None, 1]))
        def body(index, gains, coefficients, alphas, *current_states):
            gain, coeff, alpha, next_states = self.cell(erb_sequence[:, index], complex_sequence[:, index], list(current_states), training=training)
            return index + 1, gains.write(index, gain), coefficients.write(index, coeff), alphas.write(index, alpha), *next_states
        result = tf.while_loop(lambda index, *_: index < frames, body, (tf.constant(0), gains, coefficients, alphas, *states), parallel_iterations=1)
        _, gains, coefficients, alphas, *_ = result
        return tf.transpose(gains.stack(), [1, 0, 2]), tf.transpose(coefficients.stack(), [1, 0, 2, 3, 4]), tf.transpose(alphas.stack(), [1, 0, 2])

def require_float32(name, value):
    value = tf.convert_to_tensor(value)
    tf.debugging.assert_type(value, tf.float32, message=f"{name} phải là float32; complex STFT chỉ tồn tại trong DSP/loss")
    return value

class DeepFilterNetFrameCore(tf.keras.Model):
    def __init__(self, trained):
        super().__init__(name="deepfilternet_u55_frame_core", dtype="float32", autocast=False)
        self.cell = trained.cell
    def call(self, inputs):
        states = [require_float32(name, inputs[name]) for name in STATE_NAMES]
        gain, coefficients, alpha, next_states = self.cell(require_float32("erb_feature", inputs["erb_feature"]), require_float32("complex_feature", inputs["complex_feature"]), states, training=False)
        output = {"gain_bins": gain, "df_coefficients": coefficients, "alpha": alpha}
        output.update({f"next_{name}": value for name, value in zip(STATE_NAMES, next_states)})
        return output

def initial_states(batch=1):
    return {**{name: np.zeros([batch, ENC_STATE_SIZE if name.startswith("enc_") else DF_STATE_SIZE], np.float32) for name in STATE_NAMES[:5]}, "erb_conv_cache": np.zeros([batch, ERB_BANDS], np.float32), "complex_conv_cache": np.zeros([batch, 1, DF_BINS, 2], np.float32)}

MODEL = DeepFilterNetTrain()
dummy_erb, dummy_complex = tf.zeros([1, 1, ERB_BANDS]), tf.zeros([1, 1, DF_BINS, 2])
MODEL(dummy_erb, dummy_complex)
FRAME_CORE = DeepFilterNetFrameCore(MODEL)
probe_outputs = FRAME_CORE({"erb_feature": np.zeros([1, ERB_BANDS], np.float32), "complex_feature": np.zeros([1, DF_BINS, 2], np.float32), **initial_states()})
assert probe_outputs["gain_bins"].shape == (1, FFT_BINS)
assert probe_outputs["df_coefficients"].shape == (1, DF_TAPS, DF_BINS, 2)
assert probe_outputs["alpha"].shape == (1, 1)
print({"states": {name: tuple(value.shape) for name, value in initial_states().items()}, "outputs": {name: tuple(value.shape) for name, value in probe_outputs.items()}})


{'states': {'enc_state_0': (1, 320), 'enc_state_1': (1, 320), 'enc_state_2': (1, 320), 'df_state_0': (1, 320), 'df_state_1': (1, 320), 'erb_conv_cache': (1, 32), 'complex_conv_cache': (1, 1, 144, 2)}, 'outputs': {'gain_bins': (1, 257), 'df_coefficients': (1, 5, 144, 2), 'alpha': (1, 1), 'next_enc_state_0': (1, 320), 'next_enc_state_1': (1, 320), 'next_enc_state_2': (1, 320), 'next_df_state_0': (1, 320), 'next_df_state_1': (1, 320), 'next_erb_conv_cache': (1, 32), 'next_complex_conv_cache': (1, 1, 144, 2)}}


## Loss CCMSE chuẩn DeepFilterNet và train loop có log

**Mục tiêu:** Áp dụng ERB gain, causal 5-tap deep filter, alpha blend, level-normalized CCMSE và STFT consistency.

**Đầu vào:** Mixture/clean waveform batch 16 kHz.

**Đầu ra:** Checkpoint tốt nhất, history.csv và log epoch/step chi tiết.

**Quyết định kỹ thuật:** Profile full fine-tune với weight clipping tương thích Vela trước conversion INT8; debug không claim release-ready.

**Lỗi thường gặp:** Chạy cell không tuần tự, sửa shape/state contract hoặc dùng sai dataset manifest. Dừng ở assertion đầu tiên; không bỏ quality gate để tiếp tục export.

**Artifact sinh ra:** Checkpoint tốt nhất, history.csv và log epoch/step chi tiết.


In [5]:
def deep_filter(spectrum, gain_bins, coefficients, alpha):
    real_to_complex = lambda values: tf.complex(values, tf.zeros_like(values))
    masked = spectrum * real_to_complex(gain_bins)
    low = masked[:, :, :DF_BINS]
    padded = tf.pad(low, [[0, 0], [DF_TAPS - 1, 0], [0, 0]])
    coeff = tf.complex(coefficients[..., 0], coefficients[..., 1])
    filtered = tf.zeros_like(low)
    frames = tf.shape(low)[1]
    for tap in range(DF_TAPS):
        filtered += coeff[:, :, tap] * padded[:, DF_TAPS - 1 - tap:DF_TAPS - 1 - tap + frames]
    alpha_complex = real_to_complex(alpha)
    blended = (1.0 - alpha_complex) * low + alpha_complex * filtered
    return tf.concat([blended, masked[:, :, DF_BINS:]], axis=-1)

def compressed_complex(values, compression=0.6):
    magnitude = tf.abs(values)
    numerator = tf.complex(tf.pow(magnitude + 1e-8, compression), tf.zeros_like(magnitude))
    denominator = tf.complex(magnitude + 1e-8, tf.zeros_like(magnitude))
    return numerator * values / denominator

def loss_fn(model, mixture, clean, training):
    mixture_spectrum, clean_spectrum = stft(mixture), stft(clean)
    erb_sequence, complex_sequence = make_features(mixture_spectrum)
    tf.debugging.assert_type(mixture_spectrum, tf.complex64, message="STFT mixture phải là complex64 trước DSP")
    tf.debugging.assert_type(clean_spectrum, tf.complex64, message="STFT clean phải là complex64 trước DSP")
    tf.debugging.assert_type(erb_sequence, tf.float32, message="ERB feature phải là float32")
    tf.debugging.assert_type(complex_sequence, tf.float32, message="Complex feature phải là real/imag float32")
    gain_bins, coefficients, alpha = model(erb_sequence, complex_sequence, training=training)
    enhanced = deep_filter(mixture_spectrum, gain_bins, coefficients, alpha)
    target_rms = tf.sqrt(tf.reduce_mean(tf.square(clean), axis=1, keepdims=True) + 1e-8)
    scale = tf.complex(target_rms[:, :, None], tf.zeros_like(target_rms[:, :, None]))
    ccmse = tf.reduce_mean(tf.square(tf.abs(compressed_complex(enhanced / scale) - compressed_complex(clean_spectrum / scale))))
    consistent = stft(istft(enhanced))
    consistent = consistent[:, :tf.shape(clean_spectrum)[1]]
    consistency = tf.reduce_mean(tf.square(tf.abs(compressed_complex(consistent / scale) - compressed_complex(clean_spectrum / scale))))
    total = ccmse + 0.1 * consistency
    tf.debugging.assert_all_finite(total, "loss có NaN/Inf")
    return total

def evaluate_loss(limit, label, crop_samples=None):
    values = []
    items = len(val_pairs) if limit is None else min(len(val_pairs), limit)
    total_batches = math.ceil(items / RUN_CONFIG["batch"])
    started = time.monotonic()

    print({"validation_start": label, "pairs": items, "batches": total_batches})
    for index, (mixture, clean) in enumerate(
        make_dataset(
            val_pairs, training=False, limit=limit, crop_samples=crop_samples
        ),
        1,
    ):
        values.append(float(loss_fn(MODEL, mixture, clean, False)))
        if index == 1 or index % 25 == 0 or index == total_batches:
            print(
                f"{label} batch={index:04d}/{total_batches} "
                f"loss={np.mean(values[-25:]):.5f} "
                f"elapsed_s={time.monotonic()-started:.1f}"
            )

    if not values:
        raise RuntimeError("Validation stream rỗng")
    return float(np.mean(values))

def calibrate_qat_ranges(max_batches):
    if max_batches < 1:
        return

    crop_samples = RUN_CONFIG["qat_crop_samples"]
    set_qat_observer(MODEL, True)
    refresh_kernel_ranges(MODEL)

    started = time.monotonic()
    for index, (mixture, clean) in enumerate(
        make_dataset(
            train_pairs,
            training=False,
            limit=max_batches * RUN_CONFIG["batch"],
            crop_samples=crop_samples,
        ),
        1,
    ):
        _ = loss_fn(MODEL, mixture, clean, False)
        print(
            f"qat_calibration batch={index:02d}/{max_batches} "
            f"elapsed_s={time.monotonic()-started:.1f}"
        )

    set_qat_observer(MODEL, False)
    print({
        "qat_calibration_complete": True,
        "batches": max_batches,
        "crop_samples": crop_samples,
        "elapsed_s": round(time.monotonic() - started, 1),
    })


def sample_qat_saturation():
    set_qat_saturation_sampling(MODEL, True)

    mixture, _ = next(iter(make_dataset(
        val_pairs,
        training=False,
        limit=1,
        crop_samples=RUN_CONFIG["qat_crop_samples"],
    )))
    spectrum = stft(mixture)
    erb_sequence, complex_sequence = make_features(spectrum)

    # Chỉ audit một frame; không chạy loss/STFT-consistency cả clip.
    _ = MODEL(
        erb_sequence[:, :1],
        complex_sequence[:, :1],
        training=False,
    )

    set_qat_saturation_sampling(MODEL, False)
    return qat_range_report(MODEL)

def train_stage(stage, max_epochs, learning_rate, min_learning_rate, early_start, patience, restore_path=None):
    qat = stage == "qat"
    set_qat_enabled(MODEL, qat)
    optimizer = tf.keras.optimizers.Adam(learning_rate)
    checkpoint = tf.train.Checkpoint(model=MODEL, optimizer=optimizer)
    stage_dir = RUN_DIR / f"checkpoint_{stage}"
    manager = tf.train.CheckpointManager(checkpoint, str(stage_dir), max_to_keep=1)
    if restore_path:
        tf.train.Checkpoint(model=MODEL).restore(restore_path).expect_partial()
        print({"stage": stage, "restored": restore_path})
    if qat:
        calibrate_qat_ranges(RUN_CONFIG["qat_calibration_batches"])
    train_crop_samples = RUN_CONFIG["qat_crop_samples"] if qat else CROP_SAMPLES
    train_ds = iter(make_dataset(
        train_pairs, training=True, crop_samples=train_crop_samples
    ))
    steps_per_epoch = RUN_CONFIG[f"{stage}_steps_per_virtual_epoch"]
    best, bad_epochs, plateau_epochs, history, best_path = math.inf, 0, 0, [], None
    stage_started, completed_epoch_seconds = time.monotonic(), []
    print({"stage_start": stage, "qat": qat, "virtual_epochs_max": max_epochs, "steps_per_virtual_epoch": steps_per_epoch, "validation_every": RUN_CONFIG["validation_every"], "validation_pairs": RUN_CONFIG["val_items"]})
    @tf.function
    def step(mixture, clean):
        with tf.GradientTape() as tape:
            value = loss_fn(MODEL, mixture, clean, True)
        gradients = tape.gradient(value, MODEL.trainable_variables)
        optimizer.apply_gradients((gradient, variable) for gradient, variable in zip(gradients, MODEL.trainable_variables) if gradient is not None)
        return value
    for epoch in range(1, max_epochs + 1):
        values, step_seconds, data_wait_seconds, train_compute_seconds, started = [], [], [], [], time.monotonic()
        if qat:
            refresh_kernel_ranges(MODEL)
        print(f"{stage} epoch_start={epoch:02d}/{max_epochs} steps={steps_per_epoch} lr={float(optimizer.learning_rate.numpy()):.7g}")
        for index in range(1, steps_per_epoch + 1):
            data_started = time.monotonic()
            batch = next(train_ds)
            data_wait_seconds.append(time.monotonic() - data_started)
            step_started = time.monotonic()
            values.append(float(step(*batch)))
            train_compute_seconds.append(time.monotonic() - step_started)
            if index > 1:
                step_seconds.append(train_compute_seconds[-1])  # step 1 includes tf.function trace
            if index == 1 or index % 25 == 0:
                print(f"{stage} epoch={epoch:02d} step={index:04d}/{steps_per_epoch} loss={np.mean(values[-25:]):.5f} elapsed_s={time.monotonic()-started:.1f}")
        validate_now = epoch % RUN_CONFIG["validation_every"] == 0 or epoch == max_epochs
        epoch_seconds = time.monotonic() - started
        completed_epoch_seconds.append(epoch_seconds)
        eta_seconds = np.mean(completed_epoch_seconds) * (max_epochs - epoch)
        if not validate_now:
            history.append({"stage": stage, "epoch": epoch, "train_loss": float(np.mean(values)), "fixed_val_loss": None, "full_validation_on_champion": "deferred_to_frozen_test", "learning_rate": float(optimizer.learning_rate.numpy()), "cached_step_s": float(np.mean(step_seconds)) if step_seconds else None, "data_wait_s": float(np.mean(data_wait_seconds)), "train_compute_s": float(np.mean(train_compute_seconds))})
            print({"epoch_end": epoch, "stage": stage, "train_loss": float(np.mean(values)), "validation": "skipped", "next_validation_epoch": epoch + 1, "epoch_s": round(epoch_seconds, 1), "stage_s": round(time.monotonic()-stage_started, 1), "eta_s": round(eta_seconds, 1), "data_wait_s": round(float(np.mean(data_wait_seconds)), 3), "train_compute_s": round(float(np.mean(train_compute_seconds)), 3)})
            continue
        fixed_val = evaluate_loss(
            RUN_CONFIG["val_items"],
            f"{stage}_validation",
            crop_samples=train_crop_samples,
        )
        saturation = None
        if qat:
            saturation = sample_qat_saturation()
            (RUN_DIR / f"qat_saturation_epoch_{epoch:02d}.json").write_text(json.dumps(saturation, indent=2), encoding="utf-8")
        improved = fixed_val < best
        full_val = "deferred_to_frozen_test"
        row = {"stage": stage, "epoch": epoch, "train_loss": float(np.mean(values)), "fixed_val_loss": fixed_val, "full_validation_on_champion": full_val, "learning_rate": float(optimizer.learning_rate.numpy()), "qat_saturation_sampled": qat, "cached_step_s": float(np.mean(step_seconds)) if step_seconds else None, "data_wait_s": float(np.mean(data_wait_seconds)), "train_compute_s": float(np.mean(train_compute_seconds))}
        history.append(row); print({"epoch_end": epoch, "stage": stage, "train_loss": row["train_loss"], "fixed_val_loss": fixed_val, "champion": improved, "epoch_s": round(time.monotonic()-started, 1), "stage_s": round(time.monotonic()-stage_started, 1), "eta_s": round(eta_seconds, 1), "data_wait_s": round(float(np.mean(data_wait_seconds)), 3), "train_compute_s": round(float(np.mean(train_compute_seconds)), 3)})
        if improved:
            best, bad_epochs, plateau_epochs = fixed_val, 0, 0
            best_path = manager.save(checkpoint_number=epoch)
            print({"checkpoint_champion": best_path, "stage": stage, "epoch": epoch, "fixed_val_loss": best})
        else:
            bad_epochs, plateau_epochs = bad_epochs + 1, plateau_epochs + 1
            if plateau_epochs >= 3 and float(optimizer.learning_rate.numpy()) > min_learning_rate:
                optimizer.learning_rate.assign(max(min_learning_rate, float(optimizer.learning_rate.numpy()) * 0.5)); plateau_epochs = 0
                print({"stage": stage, "lr_reduced": float(optimizer.learning_rate.numpy())})
        if epoch >= early_start and bad_epochs >= patience:
            print({"stage": stage, "early_stop": True, "epoch": epoch, "best_fixed_val": best})
            break
    if best_path is None:
        raise RuntimeError(f"{stage} không tạo checkpoint champion")
    checkpoint.restore(best_path).expect_partial()
    pd.DataFrame(history).to_csv(RUN_DIR / f"history_{stage}.csv", index=False)
    validated = [row for row in history if row["fixed_val_loss"] is not None]
    manifest = {"model": MODEL_NAME, "variant": MODEL_VARIANT, "stage": stage, "best_epoch": int(min(validated, key=lambda row: row["fixed_val_loss"])["epoch"]), "best_val_loss": best, "checkpoint_path": best_path, "steps_per_epoch": steps_per_epoch, "qat": qat}
    (RUN_DIR / f"checkpoint_manifest_{stage}.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return best_path, history

def train_model():
    fp32_path, fp32_history = train_stage("fp32", RUN_CONFIG["fp32_max_epochs"], 3e-4, 2e-5, FP32_EARLY_STOP_START, FP32_EARLY_STOP_PATIENCE)
    qat_path, qat_history = train_stage("qat", RUN_CONFIG["qat_max_epochs"], 3e-5, 5e-6, QAT_EARLY_STOP_START, QAT_EARLY_STOP_PATIENCE, restore_path=fp32_path)
    fp32_steps = [row["cached_step_s"] for row in fp32_history if row.get("cached_step_s")]
    qat_steps = [row["cached_step_s"] for row in qat_history if row.get("cached_step_s")]
    speed = {"fp32_cached_step_s": float(np.mean(fp32_steps)) if fp32_steps else None, "qat_cached_step_s": float(np.mean(qat_steps)) if qat_steps else None, "qat_to_fp32_limit": 3.0}
    speed["qat_to_fp32_ratio"] = None if speed["fp32_cached_step_s"] is None or speed["qat_cached_step_s"] is None else speed["qat_cached_step_s"] / speed["fp32_cached_step_s"]
    speed["passed"] = speed["qat_to_fp32_ratio"] is None or speed["qat_to_fp32_ratio"] <= speed["qat_to_fp32_limit"]
    (RUN_DIR / "qat_speed.json").write_text(json.dumps(speed, indent=2), encoding="utf-8")
    print({"qat_speed": speed})
    if not speed["passed"]:
        print({
            "warning": (
                "QAT train chậm hơn advisory ratio; tiếp tục export. "
                "Không phải firmware release gate."
            ),
            "qat_speed": speed,
        })
    set_qat_enabled(MODEL, False)  # export graph phải sạch FakeQuant/Flex
    (RUN_DIR / "checkpoint_manifest.json").write_text(json.dumps({"model": MODEL_NAME, "variant": MODEL_VARIANT, "fp32": str(fp32_path), "qat": str(qat_path), "qat_ranges": "qat_ranges.json"}, indent=2), encoding="utf-8")
    (RUN_DIR / "qat_ranges.json").write_text(json.dumps(qat_range_report(MODEL), indent=2), encoding="utf-8")
    print({"training_complete": True, "fp32_checkpoint": fp32_path, "qat_checkpoint": qat_path})

if RUN_CONFIG["train"]:
    train_model()

def restore_frozen_checkpoint():
    manifest_path = FROZEN_CHECKPOINT_ROOT / "checkpoint_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Không tìm thấy frozen manifest: {manifest_path}")
    import hashlib
    if hashlib.sha256(manifest_path.read_bytes()).hexdigest() != FROZEN_CHECKPOINT_MANIFEST_SHA256:
        raise RuntimeError("Frozen checkpoint manifest SHA256 không khớp")
    checkpoint_path = tf.train.latest_checkpoint(str(manifest_path.parent / "checkpoint_qat"))
    if checkpoint_path is None:
        raise FileNotFoundError("Frozen checkpoint thiếu checkpoint/ckpt-* files")
    tf.train.Checkpoint(model=MODEL).restore(checkpoint_path).expect_partial()
    print({"restored_frozen_checkpoint": checkpoint_path})

if RUN_CONFIG["final"]:
    restore_frozen_checkpoint()


{'qat_enabled': False, 'fake_quantizers': 183, 'observer': False}
{'stage_start': 'fp32', 'qat': False, 'virtual_epochs_max': 22, 'steps_per_virtual_epoch': 400, 'validation_every': 2, 'validation_pairs': 256}
fp32 epoch_start=01/22 steps=400 lr=0.0003
fp32 epoch=01 step=0001/400 loss=32.85212 elapsed_s=33.6
fp32 epoch=01 step=0025/400 loss=6.20775 elapsed_s=78.2
fp32 epoch=01 step=0050/400 loss=3.33853 elapsed_s=125.9
fp32 epoch=01 step=0075/400 loss=16.32089 elapsed_s=174.0
fp32 epoch=01 step=0100/400 loss=3.64318 elapsed_s=222.4
fp32 epoch=01 step=0125/400 loss=3.20821 elapsed_s=269.6
fp32 epoch=01 step=0150/400 loss=6.39628 elapsed_s=317.0
fp32 epoch=01 step=0175/400 loss=5.98082 elapsed_s=363.8
fp32 epoch=01 step=0200/400 loss=3.66180 elapsed_s=410.0
fp32 epoch=01 step=0225/400 loss=14.07203 elapsed_s=456.0
fp32 epoch=01 step=0250/400 loss=5.13925 elapsed_s=501.7
fp32 epoch=01 step=0275/400 loss=4.72712 elapsed_s=547.6
fp32 epoch=01 step=0300/400 loss=4.53708 elapsed_s=593.4
fp32 

## QAT range audit trước export

**Mục tiêu:** Lưu range và saturation fake-quant của champion QAT; không cho export trace mang FakeQuant.

**Đầu vào:** Champion QAT trong RUN_DIR.

**Đầu ra:** qat_ranges.json và qat_smoke_report.json.

**Quyết định kỹ thuật:** Range chỉ để audit training. Frame-core export luôn tắt QAT và được PTQ full INT8 riêng.

**Lỗi thường gặp:** Chạy cell không tuần tự, sửa shape/state contract hoặc dùng sai dataset manifest. Dừng ở assertion đầu tiên; không bỏ quality gate để tiếp tục export.

**Artifact sinh ra:** qat_ranges.json và qat_smoke_report.json.


In [6]:
if RUN_CONFIG["train"]:
    qat_ranges = qat_range_report(MODEL)
    max_saturation = max((record["saturation_ratio"] for record in qat_ranges.values()), default=0.0)
    qat_smoke = {"fake_quantizers": len(qat_ranges), "max_saturation_ratio": max_saturation, "export_qat_enabled": False}
    (RUN_DIR / "qat_smoke_report.json").write_text(json.dumps(qat_smoke, indent=2), encoding="utf-8")
    print({"qat_smoke": qat_smoke, "range_artifact": str(RUN_DIR / "qat_ranges.json")})


{'qat_smoke': {'fake_quantizers': 183, 'max_saturation_ratio': 0.14814814814814814, 'export_qat_enabled': False}, 'range_artifact': '/kaggle/working/deepfilternet_u55_full_seed20260805/qat_ranges.json'}


## Export INT8 có tensor quantization contract

**Mục tiêu:** Export SavedModel signature, convert full INT8 và ghi scale/zero-point cho mọi input/output firmware dùng.

**Đầu vào:** Frame core, calibration stream validation và checkpoint tốt nhất.

**Đầu ra:** TFLite pre-Vela, model headers, tensor_contract.json và export_metadata.json.

**Quyết định kỹ thuật:** Không suy đoán Q-format trong firmware: chỉ contract sinh từ TFLite mới là nguồn đúng.

**Lỗi thường gặp:** Chạy cell không tuần tự, sửa shape/state contract hoặc dùng sai dataset manifest. Dừng ở assertion đầu tiên; không bỏ quality gate để tiếp tục export.

**Artifact sinh ra:** TFLite pre-Vela, model headers, tensor_contract.json và export_metadata.json.


In [7]:
PACKAGE = RUN_DIR / "integration_package"
PACKAGE.mkdir(parents=True, exist_ok=True)
if (RUN_DIR / "qat_ranges.json").is_file():
    shutil.copy2(RUN_DIR / "qat_ranges.json", PACKAGE / "qat_ranges.json")
GOLDEN_MIN_FRAMES = 100

def representative_frames(max_frames=128):
    samples = []
    for mixture, _ in make_dataset(val_pairs, training=False, limit=16).unbatch().batch(1):
        spectrum = stft(mixture)
        erb_sequence, complex_sequence = make_features(spectrum)
        state = initial_states()
        for index in range(int(erb_sequence.shape[1])):
            sample = {"erb_feature": erb_sequence[:, index].numpy().astype(np.float32), "complex_feature": complex_sequence[:, index].numpy().astype(np.float32), **state}
            samples.append(sample)
            result = FRAME_CORE(sample)
            state = {name: result[f"next_{name}"].numpy().astype(np.float32) for name in STATE_NAMES}
            if len(samples) >= max_frames:
                return samples
    return samples

def sha256_file(path):
    import hashlib
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def c_identifier(name):
    return "".join(character if character.isalnum() else "_" for character in name).upper()

def detail_record(detail):
    scale, zero_point = detail.get("quantization", (0.0, 0))
    return {
        "name": detail["name"], "shape": [int(value) for value in detail["shape"]],
        "dtype": np.dtype(detail["dtype"]).name, "index": int(detail["index"]),
        "scale": float(scale), "zero_point": int(zero_point),
    }

def normalize_details(details):
    return details if isinstance(details, dict) else {detail["name"].split(":")[0]: detail for detail in details}

def quantize_tensor(values, detail):
    scale, zero_point = detail["quantization"]
    if detail["dtype"] != np.int8 or scale <= 0:
        raise ValueError(f"Tensor không phải INT8 quantized: {detail['name']}")
    return np.clip(np.rint(np.asarray(values, np.float32) / scale + zero_point), -128, 127).astype(np.int8)

def dequantize_tensor(values, detail):
    scale, zero_point = detail["quantization"]
    if detail["dtype"] != np.int8 or scale <= 0:
        raise ValueError(f"Tensor không phải INT8 quantized: {detail['name']}")
    return (np.asarray(values, np.float32) - zero_point) * scale

def write_c_array(handle, name, values):
    values = np.asarray(values)
    types = {np.dtype("float32"): "float", np.dtype("int8"): "int8_t", np.dtype("int16"): "int16_t", np.dtype("uint8"): "uint8_t"}
    c_type = types.get(values.dtype)
    if c_type is None:
        raise TypeError(f"Không hỗ trợ C dtype {values.dtype}")
    flat = values.reshape(-1)
    if values.dtype == np.float32:
        payload = ", ".join(f"{float(value):.9g}f" for value in flat)
    else:
        payload = ", ".join(str(int(value)) for value in flat)
    handle.write(f"alignas(16) static const {c_type} {name}[{len(flat)}] = {{{payload}}};\n")
    handle.write(f"static const size_t {name}_len = {len(flat)};\n")

def write_model_data(stem, model_path, symbol):
    data = Path(model_path).read_bytes()
    header, source = Path(stem).with_suffix(".h"), Path(stem).with_suffix(".c")
    guard = c_identifier(header.stem)
    header.write_text(f"#pragma once\n#include <stdint.h>\n#include <stddef.h>\nextern const unsigned char {symbol}[];\nextern const size_t {symbol}_len;\n", encoding="utf-8")
    with source.open("w", encoding="utf-8") as handle:
        handle.write(f"#include \"{header.name}\"\n#include <stdalign.h>\n")
        handle.write(f"alignas(16) const unsigned char {symbol}[] = {{")
        handle.write(", ".join(str(value) for value in data))
        handle.write(f"}};\nconst size_t {symbol}_len = {len(data)};\n")
    return header, source

def concrete_core_signature():
    signature = {"erb_feature": tf.TensorSpec([1, ERB_BANDS], tf.float32, "erb_feature"), "complex_feature": tf.TensorSpec([1, DF_BINS, 2], tf.float32, "complex_feature")}
    signature.update({name: tf.TensorSpec(value.shape, tf.float32, name) for name, value in initial_states().items()})
    @tf.function(input_signature=[signature])
    def serve(inputs):
        return FRAME_CORE(inputs)
    return serve.get_concrete_function()

def export_full_int8():
    samples = representative_frames(max_frames=512 if RUN_PROFILE == "full" else 16)
    if not samples:
        raise RuntimeError("Calibration stream rỗng")
    saved_model_dir = PACKAGE / "saved_model_for_int8"
    if saved_model_dir.exists():
        shutil.rmtree(saved_model_dir)
    concrete = concrete_core_signature()
    export_module = tf.Module()
    for index, variable in enumerate(FRAME_CORE.variables):
        setattr(export_module, f"variable_{index}", variable.value if hasattr(variable, "value") else variable)
    export_module.serve = concrete
    tf.saved_model.save(export_module, str(saved_model_dir), signatures={"serving_default": concrete})
    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir), signature_keys=["serving_default"])
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = lambda: ({key: tf.convert_to_tensor(value) for key, value in sample.items()} for sample in samples)
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    pre_vela_path = PACKAGE / "deepfilternet_u55_int8.tflite"
    pre_vela_path.write_bytes(converter.convert())
    interpreter = tf.lite.Interpreter(model_path=str(pre_vela_path))
    interpreter.allocate_tensors()
    runner = interpreter.get_signature_runner("serving_default")
    inputs, outputs = normalize_details(runner.get_input_details()), normalize_details(runner.get_output_details())
    contract = {"schema_version": 1, "model": MODEL_NAME, "inputs": {name: detail_record(detail) for name, detail in inputs.items()}, "outputs": {name: detail_record(detail) for name, detail in outputs.items()}}
    contract["state_mapping"] = [{"input": name, "output": f"next_{name}", "copy_mode": "raw_int8" if all(contract["inputs"][name][key] == contract["outputs"][f"next_{name}"][key] for key in ("shape", "dtype", "scale", "zero_point")) else "requantize", "reset": "zero_on_boot_or_stream_discontinuity"} for name in STATE_NAMES]
    for name in ("gain_bins", "df_coefficients", "alpha"):
        if name not in contract["outputs"]:
            raise RuntimeError(f"TFLite signature thiếu output {name}")
    if any(record["dtype"] != "int8" or record["scale"] <= 0 for group in (contract["inputs"], contract["outputs"]) for record in group.values()):
        raise RuntimeError("Pre-Vela model không full INT8")
    operators = interpreter._get_ops_details()
    if any("Flex" in str(detail) for detail in operators):
        raise RuntimeError("Pre-Vela graph chứa Flex operator")
    (PACKAGE / "tensor_contract.json").write_text(json.dumps(contract, indent=2), encoding="utf-8")
    write_model_data(PACKAGE / "deepfilternet_u55_pre_vela_model_data", pre_vela_path, "deepfilternet_u55_pre_vela_model")
    metadata = {"pre_vela": {"path": pre_vela_path.name, "bytes": pre_vela_path.stat().st_size, "sha256": sha256_file(pre_vela_path)}, "calibration_frames": len(samples), "operators": [detail["op_name"] for detail in operators]}
    (PACKAGE / "export_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return pre_vela_path, contract

if RUN_CONFIG["export"]:
    PRE_VELA_PATH, TENSOR_CONTRACT = export_full_int8()
elif RUN_CONFIG["final"]:
    PRE_VELA_PATH = FROZEN_PACKAGE_ROOT / "deepfilternet_u55_int8.tflite"
    contract_path = FROZEN_PACKAGE_ROOT / "tensor_contract.json"
    if not PRE_VELA_PATH.is_file() or not contract_path.is_file():
        raise FileNotFoundError("final cần attach frozen integration_package chứa pre-Vela model và tensor contract")
    TENSOR_CONTRACT = json.loads(contract_path.read_text(encoding="utf-8"))
else:
    PRE_VELA_PATH, TENSOR_CONTRACT = None, None
print({"export": str(PRE_VELA_PATH) if PRE_VELA_PATH else "disabled", "tensor_contract": TENSOR_CONTRACT is not None})


INFO:tensorflow:Assets written to: /kaggle/working/deepfilternet_u55_full_seed20260805/integration_package/saved_model_for_int8/assets


INFO:tensorflow:Assets written to: /kaggle/working/deepfilternet_u55_full_seed20260805/integration_package/saved_model_for_int8/assets
W0000 00:00:1785890884.579686      23 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1785890884.579763      23 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1785890884.789654      23 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.

{'export': '/kaggle/working/deepfilternet_u55_full_seed20260805/integration_package/deepfilternet_u55_int8.tflite', 'tensor_contract': True}


## Vela U55 strict compile và operator audit

**Mục tiêu:** Compile TFLite INT8 bằng Ethos-U Vela và dừng nếu còn CPU/Flex/FP16 operator.

**Đầu vào:** Pre-Vela model và VELA_ACCELERATOR=ethos-u55-256.

**Đầu ra:** Post-Vela model, Vela report và model C header aligned 16 byte.

**Quyết định kỹ thuật:** `CPU operators = 0` là release gate; không được thay bằng fallback CPU.

**Lỗi thường gặp:** Chạy cell không tuần tự, sửa shape/state contract hoặc dùng sai dataset manifest. Dừng ở assertion đầu tiên; không bỏ quality gate để tiếp tục export.

**Artifact sinh ra:** Post-Vela model, Vela report và model C header aligned 16 byte.


In [8]:
def compile_vela(pre_vela_path):
    config_path = PACKAGE / "vela_m55m1.ini"
    config_path.write_text("""; M55M1: U55-256 @ 220 MHz; APROM=2 MiB, SRAM total=1.5 MiB.
[System_Config.M55M1_U55_256]
core_clock=220e6
axi0_port=Sram
axi1_port=OnChipFlash
Sram_clock_scale=1.0
Sram_burst_length=32
Sram_read_latency=1
Sram_write_latency=1
OnChipFlash_clock_scale=0.25
OnChipFlash_burst_length=32
OnChipFlash_read_latency=16
OnChipFlash_write_latency=16

[Memory_Mode.M55M1_Shared_SRAM]
const_mem_area=Axi1
arena_mem_area=Axi0
cache_mem_area=Axi0
""", encoding="utf-8")
    variants = []
    for optimise in ("Size", "Performance"):
        vela_dir = PACKAGE / f"vela_{optimise.lower()}"
        command = ["vela", str(pre_vela_path), "--accelerator-config", VELA_ACCELERATOR, "--config", str(config_path), "--system-config", "M55M1_U55_256", "--memory-mode", "M55M1_Shared_SRAM", "--optimise", optimise, "--arena-cache-size", str(1024 * 1024), "--output-dir", str(vela_dir), "--show-cpu-operations"]
        result = subprocess.run(command, text=True, capture_output=True)
        report = result.stdout + "\n" + result.stderr
        (PACKAGE / f"vela_report_{optimise.lower()}.txt").write_text(report, encoding="utf-8")
        forbidden, candidates = [token for token in ("Flex", "FP16") if token in report], list(vela_dir.glob("*_vela.tflite"))
        if result.returncode == 0 and "CPU operators = 0" in report and not forbidden and len(candidates) == 1:
            variants.append({"optimise": optimise, "path": candidates[0], "bytes": candidates[0].stat().st_size, "report": report})
        else:
            print({"vela_variant_failed": optimise, "return": result.returncode, "forbidden": forbidden, "models": [str(path) for path in candidates]})
    if not variants:
        raise RuntimeError("M55M1 Vela gate fail; xem vela_report_size.txt và vela_report_performance.txt")
    chosen = min(variants, key=lambda item: item["bytes"])
    output = PACKAGE / "deepfilternet_u55_int8_vela.tflite"
    shutil.copy2(chosen["path"], output)
    (PACKAGE / "vela_report.txt").write_text(chosen["report"], encoding="utf-8")
    selection = {"selected": chosen["optimise"], "variants": [{"optimise": item["optimise"], "path": str(item["path"]), "bytes": item["bytes"]} for item in variants], "board_e2e_gate_us": 16000, "board_measurement_required": True}
    (PACKAGE / "vela_selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
    write_model_data(PACKAGE / "deepfilternet_u55_vela_model_data", output, "deepfilternet_u55_vela_model")
    return output

VELA_PATH = compile_vela(PRE_VELA_PATH) if RUN_CONFIG["export"] and PRE_VELA_PATH else None
print({"vela_model": str(VELA_PATH) if VELA_PATH else "disabled", "required_cpu_operators": 0})


{'vela_model': '/kaggle/working/deepfilternet_u55_full_seed20260805/integration_package/deepfilternet_u55_int8_vela.tflite', 'required_cpu_operators': 0}


## DSP/MVE constants và firmware interface

**Mục tiêu:** Sinh C headers cho ERB, windows, gain/DF contract và hướng dẫn M55 Helium.

**Đầu vào:** ERB matrices, sqrt-Hann window, tensor quantization contract và architecture constants.

**Đầu ra:** dsp_constants.h, deepfilternet_u55_contract.h và README_integration.md.

**Quyết định kỹ thuật:** CPU chỉ thực hiện DSP: FFT/WOLA, complex multiply-accumulate, dequantize/quantize/state copy. NPU chạy toàn bộ graph neural.

**Lỗi thường gặp:** Chạy cell không tuần tự, sửa shape/state contract hoặc dùng sai dataset manifest. Dừng ở assertion đầu tiên; không bỏ quality gate để tiếp tục export.

**Artifact sinh ra:** dsp_constants.h, deepfilternet_u55_contract.h và README_integration.md.


In [9]:
def export_firmware_headers(contract):
    with (PACKAGE / "dsp_constants.h").open("w", encoding="utf-8") as handle:
        handle.write("#pragma once\n#include <stdint.h>\n#include <stddef.h>\n#include <stdalign.h>\n")
        handle.write(f"#define DFN_SAMPLE_RATE {SAMPLE_RATE}\n#define DFN_FRAME_LENGTH {FRAME_LENGTH}\n#define DFN_FRAME_STEP {FRAME_STEP}\n#define DFN_FFT_BINS {FFT_BINS}\n#define DFN_ERB_BANDS {ERB_BANDS}\n#define DFN_DF_BINS {DF_BINS}\n#define DFN_DF_TAPS {DF_TAPS}\n")
        write_c_array(handle, "dfn_erb_analysis_257x32", ERB_ANALYSIS_NP)
        write_c_array(handle, "dfn_erb_expansion_32x257", ERB_EXPANSION_NP)
        write_c_array(handle, "dfn_sqrt_hann_512", sqrt_hann(FRAME_LENGTH).numpy().astype(np.float32))
    with (PACKAGE / "deepfilternet_u55_contract.h").open("w", encoding="utf-8") as handle:
        handle.write("#pragma once\n#include <stdint.h>\n#include <stddef.h>\n")
        for direction in ("inputs", "outputs"):
            for logical_name, record in contract[direction].items():
                prefix = c_identifier(f"DFN_{direction}_{logical_name}")
                handle.write(f"#define {prefix}_SCALE {record['scale']:.12g}f\n#define {prefix}_ZERO_POINT {record['zero_point']}\n")
        handle.write("/* MVE path: arm_cfft_q15, arm_cmplx_mult_real_q15, fused 5-tap complex MAC. */\n")
        handle.write("#if defined(__ARM_FEATURE_MVE)\n#define DFN_USE_HELIUM_MVE 1\n#else\n#define DFN_USE_HELIUM_MVE 0\n#endif\n")
    (PACKAGE / "README_integration.md").write_text("""# DeepFilterNet U55 firmware contract

1. Reset every state input to zero at boot or stream discontinuity.
2. Quantize feature/state using `deepfilternet_u55_contract.h`; invoke Vela model; follow `tensor_contract.json.state_mapping`: raw copy is valid only when `copy_mode` is `raw_int8`, otherwise requantize next state.
3. Dequantize `gain_bins`, `df_coefficients`, `alpha` with their own scale/zero-point. Never treat INT8 gain as Q7 by assumption.
4. M55 DSP: apply 257 bin gains, retain five low-band spectra, execute causal complex 5-tap MAC to bin 143, alpha blend, IFFT/WOLA.
5. Build MVE path when `__ARM_FEATURE_MVE` is set; retain scalar reference only for parity. Log fft_us/npu_us/df_us/wola_us/e2e_us.
6. Run golden vectors before audio release. `board_mve_parity_pending` is not release-ready.
7. `deepfilternet_u55_postprocess.c` là reference chính xác cho gain/DF/alpha; thay inner loop bằng MVE sau khi golden parity pass.
8. Compile exactly one model-data source: `deepfilternet_u55_vela_model_data.c`; include its `.h` everywhere else. Inspect the final linker map: `.text + .rodata + model + DSP constants` must stay at or below 1.8 MiB.
9. Fill `board_gate.json` from `board_gate_template.json` after target run. Release needs `mve_scalar_max_lsb_q15 <= 1` and `e2e_us <= 16000`.
""", encoding="utf-8")
    (PACKAGE / "board_gate_template.json").write_text(json.dumps({"fft_us": None, "npu_us": None, "df_us": None, "wola_us": None, "e2e_us": None, "mve_scalar_max_lsb_q15": None, "linker_flash_bytes": None, "linker_sram_bytes": None, "gate": {"e2e_us_max": 16000, "mve_scalar_max_lsb_q15": 1, "flash_release_budget_bytes": FLASH_RELEASE_BUDGET_BYTES}}, indent=2), encoding="utf-8")
    (PACKAGE / "deepfilternet_u55_postprocess.h").write_text("""#pragma once
#include <stdint.h>
void dfn_apply_postprocess(const float *spectrum_257x2, const float *gain_257, const float *coeff_5x144x2, float alpha, float *history_5x144x2, float *output_257x2);
""", encoding="utf-8")
    (PACKAGE / "deepfilternet_u55_postprocess.c").write_text("""#include \"deepfilternet_u55_postprocess.h\"
#include \"dsp_constants.h\"
#include <string.h>

/* Scalar golden reference. Replace only this inner loop with Helium after golden parity passes. */
void dfn_apply_postprocess(const float *x, const float *gain, const float *coeff, float alpha, float *history, float *y) {
    for (int k = 0; k < DFN_FFT_BINS; ++k) { y[2*k] = x[2*k] * gain[k]; y[2*k+1] = x[2*k+1] * gain[k]; }
    for (int tap = DFN_DF_TAPS - 1; tap > 0; --tap) memcpy(&history[2*tap*DFN_DF_BINS], &history[2*(tap-1)*DFN_DF_BINS], 2*DFN_DF_BINS*sizeof(float));
    memcpy(history, y, 2*DFN_DF_BINS*sizeof(float));
    for (int k = 0; k < DFN_DF_BINS; ++k) {
        float fr = 0.0f, fi = 0.0f;
        for (int tap = 0; tap < DFN_DF_TAPS; ++tap) { const float *c = &coeff[2*(tap*DFN_DF_BINS+k)], *h = &history[2*(tap*DFN_DF_BINS+k)]; fr += c[0]*h[0] - c[1]*h[1]; fi += c[0]*h[1] + c[1]*h[0]; }
        y[2*k] = (1.0f-alpha)*y[2*k] + alpha*fr; y[2*k+1] = (1.0f-alpha)*y[2*k+1] + alpha*fi;
    }
}
""", encoding="utf-8")

if TENSOR_CONTRACT:
    export_firmware_headers(TENSOR_CONTRACT)


## Release gate manifest cho firmware

**Mục tiêu:** Ghi một checklist máy-đọc được để firmware CI và linker-map dùng cùng ngưỡng release.

**Đầu vào:** Tensor contract, budget APROM/SRAM và quality limits.

**Đầu ra:** release_gates.json.

**Quyết định kỹ thuật:** Kaggle chỉ xác nhận host pipeline; file board_gate.json từ M55M1 mới đóng board gate.

**Lỗi thường gặp:** Chạy cell không tuần tự, sửa shape/state contract hoặc dùng sai dataset manifest. Dừng ở assertion đầu tiên; không bỏ quality gate để tiếp tục export.

**Artifact sinh ra:** release_gates.json.


In [10]:
if TENSOR_CONTRACT:
    release_gates = {"quality": QUALITY_LIMITS, "flash_release_budget_bytes": FLASH_RELEASE_BUDGET_BYTES, "cpu_operators": 0, "board": {"e2e_us_max": 16000, "mve_scalar_max_lsb_q15": 1}, "model_data_translation_units": ["deepfilternet_u55_vela_model_data.c"]}
    (PACKAGE / "release_gates.json").write_text(json.dumps(release_gates, indent=2), encoding="utf-8")
    print({"release_gates": release_gates})


{'release_gates': {'quality': {'delta_si_sdr_db': 0.5, 'delta_pesq': 0.05, 'delta_stoi': 0.01}, 'flash_release_budget_bytes': 1887436, 'cpu_operators': 0, 'board': {'e2e_us_max': 16000, 'mve_scalar_max_lsb_q15': 1}, 'model_data_translation_units': ['deepfilternet_u55_vela_model_data.c']}}


## Golden C traces cho NPU và DSP boundary

**Mục tiêu:** Sinh năm scenario có đủ input INT8, output INT8/dequantized và state transition cho firmware regression.

**Đầu vào:** Signature runner INT8, tensor contract và waveform synthetic deterministic.

**Đầu ra:** Five headers trong golden/, golden_manifest.json và golden summary.

**Quyết định kỹ thuật:** Golden lưu boundary thật TFLite, vì đây là cách bắt sai scale/zero-point và state order trước khi chạy MVE.

**Lỗi thường gặp:** Chạy cell không tuần tự, sửa shape/state contract hoặc dùng sai dataset manifest. Dừng ở assertion đầu tiên; không bỏ quality gate để tiếp tục export.

**Artifact sinh ra:** Five headers trong golden/, golden_manifest.json và golden summary.


In [11]:
def synthetic_waveform(kind, frames=GOLDEN_MIN_FRAMES):
    length = FRAME_LENGTH + frames * FRAME_STEP
    index = np.arange(length, dtype=np.float32) / SAMPLE_RATE
    if kind == "silence": return np.zeros(length, np.float32)
    if kind == "impulse":
        output = np.zeros(length, np.float32); output[FRAME_LENGTH] = 0.5; return output
    if kind == "sine_440": return (0.25 * np.sin(2 * np.pi * 440 * index)).astype(np.float32)
    if kind == "sine_3000": return (0.25 * np.sin(2 * np.pi * 3000 * index)).astype(np.float32)
    return np.random.default_rng(SEED).normal(0.0, 0.08, length).astype(np.float32)

def trace_tflite_frames(model_path, waveform, max_frames=GOLDEN_MIN_FRAMES):
    interpreter = tf.lite.Interpreter(model_path=str(model_path)); interpreter.allocate_tensors()
    runner = interpreter.get_signature_runner("serving_default")
    input_details, output_details = normalize_details(runner.get_input_details()), normalize_details(runner.get_output_details())
    spectrum = stft(tf.convert_to_tensor(waveform[None, :], tf.float32))
    erb_sequence, complex_sequence = make_features(spectrum)
    state, records = initial_states(), []
    for index in range(min(int(erb_sequence.shape[1]), max_frames)):
        values = {"erb_feature": erb_sequence[:, index].numpy(), "complex_feature": complex_sequence[:, index].numpy(), **state}
        quantized = {name: quantize_tensor(values[name], input_details[name]) for name in input_details}
        raw = runner(**quantized)
        dequantized = {name: dequantize_tensor(raw[name], output_details[name]) for name in output_details}
        records.append({"inputs_float": values, "inputs_int8": quantized, "outputs_int8": raw, "outputs_float": dequantized})
        state = {name: dequantized[f"next_{name}"] for name in STATE_NAMES}
    return records

def export_golden_headers(model_path):
    golden_dir = PACKAGE / "golden"; golden_dir.mkdir(exist_ok=True)
    manifest = {"minimum_frames": GOLDEN_MIN_FRAMES, "scenarios": {}}
    for kind in ("silence", "impulse", "sine_440", "sine_3000", "noise"):
        records = trace_tflite_frames(model_path, synthetic_waveform(kind))
        arrays = {"input_pcm": synthetic_waveform(kind)}
        for direction in ("inputs_int8", "outputs_int8", "outputs_float"):
            for name in records[0][direction]:
                arrays[f"{kind}_{direction}_{name}"] = np.concatenate([np.asarray(record[direction][name]).reshape(-1) for record in records])
        header = golden_dir / f"golden_{kind}.h"
        with header.open("w", encoding="utf-8") as handle:
            handle.write("#pragma once\n#include <stdint.h>\n#include <stddef.h>\n#include <stdalign.h>\n")
            for name, values in arrays.items(): write_c_array(handle, c_identifier(name), np.asarray(values, np.float32 if "float" in name or name == "input_pcm" else np.int8))
        manifest["scenarios"][kind] = {"header": header.name, "frames": len(records), "sha256": sha256_file(header)}
    (golden_dir / "golden_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

if RUN_CONFIG["export"] and PRE_VELA_PATH:
    export_golden_headers(PRE_VELA_PATH)  # Host interpreter chạy pre-Vela; post-Vela cần Ethos-U delegate trên board.


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


## Parity FP32/INT8, smoke và frozen final evaluation

**Mục tiêu:** Full train chỉ tạo package firmware và quality smoke 128 pair; full frozen test được chia thành 15 shard để không timeout Kaggle.

**Đầu vào:** Full/debug dùng validation smoke. Final shard dùng checkpoint và pre-Vela package frozen; merge chỉ đọc artifact shard đã hoàn tất.

**Đầu ra:** Full tạo `metrics_smoke_*`, `parity_smoke_report.json`, `performance_full.json` và package `engineering_ready_smoke_passed`. Final merge tạo `metrics_per_sample.csv`, `metrics_summary.json`, `parity_report.json`, package audit và integration manifest.

**Quyết định kỹ thuật:** Chỉ allocate một TFLite interpreter trong mỗi smoke/shard. FP32 chạy cả sequence một lần và được đối chiếu 10 frame với `FRAME_CORE` trước khi đo metric. Final có 15 shard × 250 pair, lưu partial mỗi 16 pair và có thể resume.

**Lỗi thường gặp:** `final` cần attach checkpoint và `integration_package` frozen. `merge` cần attach đủ output 15 shard cùng model/checkpoint SHA-256; không infer hoặc train lại.

**Artifact sinh ra:** Smoke package dùng để tích hợp/board test ngay; quality release chỉ hợp lệ sau final merge và board gate M55M1.


In [12]:
def apply_dsp_frame(spectrum, gain, coefficients, alpha, history):
    masked = spectrum * gain
    history.insert(0, masked[:DF_BINS].copy()); del history[DF_TAPS:]
    while len(history) < DF_TAPS: history.append(np.zeros(DF_BINS, np.complex64))
    coeff = coefficients[..., 0] + 1j * coefficients[..., 1]
    filtered = sum(coeff[tap] * history[tap] for tap in range(DF_TAPS))
    output = masked.copy()
    output[:DF_BINS] = (1.0 - alpha) * masked[:DF_BINS] + alpha * filtered
    return output


def fp32_heads(waveform):
    spectrum = stft(tf.convert_to_tensor(waveform[None, :], tf.float32)).numpy()[0]
    erb_sequence, complex_sequence = make_features(tf.convert_to_tensor(spectrum[None, :]))
    gain, coefficients, alpha = MODEL(erb_sequence, complex_sequence, training=False)
    return spectrum, gain.numpy()[0], coefficients.numpy()[0], alpha.numpy()[0]


def run_fp32_stream(waveform):
    spectrum, gains, coefficients, alphas = fp32_heads(waveform)
    history, enhanced, traces = [], [], []
    for index in range(spectrum.shape[0]):
        gain, coeff, alpha = gains[index], coefficients[index], float(alphas[index, 0])
        enhanced.append(apply_dsp_frame(spectrum[index], gain, coeff, alpha, history))
        traces.append((gain, coeff, alpha))
    return istft(tf.convert_to_tensor(np.asarray(enhanced)[None, :], tf.complex64)).numpy()[0], traces


def assert_sequence_core_parity(waveform, max_frames=10):
    spectrum, gains, coefficients, alphas = fp32_heads(waveform)
    erb_sequence, complex_sequence = make_features(tf.convert_to_tensor(spectrum[None, :]))
    state = initial_states()
    for index in range(min(max_frames, spectrum.shape[0])):
        result = FRAME_CORE({"erb_feature": erb_sequence[:, index].numpy(),
                             "complex_feature": complex_sequence[:, index].numpy(), **state})
        np.testing.assert_allclose(gains[index], result["gain_bins"].numpy()[0], rtol=0.0, atol=1e-6)
        np.testing.assert_allclose(coefficients[index], result["df_coefficients"].numpy()[0], rtol=0.0, atol=1e-6)
        np.testing.assert_allclose(alphas[index], result["alpha"].numpy()[0], rtol=0.0, atol=1e-6)
        state = {name: result[f"next_{name}"].numpy() for name in STATE_NAMES}
    print({"sequence_frame_core_parity": "passed", "frames": min(max_frames, spectrum.shape[0])})


def create_tflite_runner(model_path):
    interpreter = tf.lite.Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    runner = interpreter.get_signature_runner("serving_default")
    return {
        "interpreter": interpreter,
        "runner": runner,
        "inputs": normalize_details(runner.get_input_details()),
        "outputs": normalize_details(runner.get_output_details()),
    }


def run_tflite_stream(runner_bundle, waveform):
    spectrum = stft(tf.convert_to_tensor(waveform[None, :], tf.float32)).numpy()[0]
    erb_sequence, complex_sequence = make_features(tf.convert_to_tensor(spectrum[None, :]))
    state, history, enhanced, traces = initial_states(), [], [], []
    for index in range(spectrum.shape[0]):
        values = {"erb_feature": erb_sequence[:, index].numpy(),
                  "complex_feature": complex_sequence[:, index].numpy(), **state}
        quantized = {name: quantize_tensor(values[name], runner_bundle["inputs"][name])
                     for name in runner_bundle["inputs"]}
        raw = runner_bundle["runner"](**quantized)
        outputs = {name: dequantize_tensor(raw[name], runner_bundle["outputs"][name])
                   for name in runner_bundle["outputs"]}
        gain, coeff, alpha = outputs["gain_bins"][0], outputs["df_coefficients"][0], float(outputs["alpha"][0, 0])
        enhanced.append(apply_dsp_frame(spectrum[index], gain, coeff, alpha, history))
        traces.append((gain, coeff, alpha))
        state = {name: outputs[f"next_{name}"] for name in STATE_NAMES}
    return istft(tf.convert_to_tensor(np.asarray(enhanced)[None, :], tf.complex64)).numpy()[0], traces


def si_sdr(reference, estimate):
    length = min(len(reference), len(estimate))
    reference, estimate = np.asarray(reference[:length], np.float64), np.asarray(estimate[:length], np.float64)
    reference -= reference.mean(); estimate -= estimate.mean()
    target = np.dot(estimate, reference) / (np.dot(reference, reference) + 1e-12) * reference
    return float(10 * np.log10((np.sum(target * target) + 1e-12) / (np.sum((estimate - target) ** 2) + 1e-12)))


def quality_metrics(reference, estimate):
    length = min(len(reference), len(estimate))
    reference, estimate = np.asarray(reference[:length], np.float32), np.asarray(estimate[:length], np.float32)
    return {"si_sdr": si_sdr(reference, estimate),
            "pesq_wb": float(pesq_score(SAMPLE_RATE, reference, estimate, "wb")),
            "stoi": float(stoi_score(reference, estimate, SAMPLE_RATE, extended=False))}


def quality_summary(frame, enforced):
    summary = frame.groupby("precision")[["si_sdr", "pesq_wb", "stoi"]].mean().to_dict()
    delta_si_sdr_db = summary["si_sdr"]["fp32"] - summary["si_sdr"]["int8"]
    delta_pesq = summary["pesq_wb"]["fp32"] - summary["pesq_wb"]["int8"]
    delta_stoi = summary["stoi"]["fp32"] - summary["stoi"]["int8"]
    return {"mean_metrics": summary, "delta_si_sdr_db": delta_si_sdr_db,
            "delta_pesq": delta_pesq, "delta_stoi": delta_stoi,
            "limits": QUALITY_LIMITS,
            "passed": bool(delta_si_sdr_db <= QUALITY_LIMITS["delta_si_sdr_db"] and
                           delta_pesq <= QUALITY_LIMITS["delta_pesq"] and
                           delta_stoi <= QUALITY_LIMITS["delta_stoi"]),
            "enforced": enforced}


def evaluation_identity():
    checkpoint_manifest = FROZEN_CHECKPOINT_ROOT / "checkpoint_manifest.json" if RUN_CONFIG["final"] else RUN_DIR / "checkpoint_manifest.json"
    return {"model_sha256": sha256_file(PRE_VELA_PATH) if PRE_VELA_PATH else None,
            "checkpoint_manifest_sha256": sha256_file(checkpoint_manifest) if checkpoint_manifest.is_file() else None}


def write_smoke_package(frame, parity, started):
    frame.to_csv(PACKAGE / "metrics_smoke_per_sample.csv", index=False)
    quality = quality_summary(frame, enforced=False)
    quality["scope"] = "smoke_128_not_release_gate"
    (PACKAGE / "metrics_smoke_summary.json").write_text(json.dumps(quality, indent=2), encoding="utf-8")
    (PACKAGE / "parity_smoke_report.json").write_text(json.dumps(parity, indent=2), encoding="utf-8")
    performance = {"items": int(frame["item"].nunique()), "elapsed_s": round(time.monotonic() - started, 1),
                   "tflite_interpreters_allocated": 1}
    (PACKAGE / "performance_full.json").write_text(json.dumps(performance, indent=2), encoding="utf-8")
    required = ["deepfilternet_u55_int8.tflite", "deepfilternet_u55_int8_vela.tflite",
                "deepfilternet_u55_vela_model_data.c", "tensor_contract.json", "dsp_constants.h",
                "deepfilternet_u55_contract.h", "deepfilternet_u55_postprocess.c", "vela_report.txt",
                "golden/golden_manifest.json", "metrics_smoke_per_sample.csv",
                "metrics_smoke_summary.json", "parity_smoke_report.json", "performance_full.json"]
    missing = [path for path in required if not (PACKAGE / path).is_file()]
    if missing:
        raise FileNotFoundError(f"Integration smoke package thiếu {missing}")
    dsp_constant_bytes = ERB_ANALYSIS_NP.nbytes + ERB_EXPANSION_NP.nbytes + sqrt_hann(FRAME_LENGTH).numpy().nbytes
    flash_estimate = VELA_PATH.stat().st_size + dsp_constant_bytes
    audit = {"required_files": len(required), "missing": missing, "vela_cpu_operators": 0,
             "flash_estimate_bytes": flash_estimate, "flash_release_budget_bytes": FLASH_RELEASE_BUDGET_BYTES,
             "flash_passed": flash_estimate <= FLASH_RELEASE_BUDGET_BYTES,
             "quality_scope": quality["scope"], "passed": not missing}
    (PACKAGE / "package_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    manifest = {"schema_version": 2, "model": MODEL_NAME, "variant": MODEL_VARIANT, "profile": RUN_PROFILE,
                "quality": quality, "parity": parity, "flash_estimate_bytes": flash_estimate,
                "release_status": "engineering_ready_smoke_passed" if not missing else "package_incomplete"}
    (PACKAGE / "integration_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print({"smoke": quality, "audit": audit, "release_status": manifest["release_status"]})


def shard_paths():
    directory = PACKAGE / "evaluation_shards"
    directory.mkdir(parents=True, exist_ok=True)
    stem = f"metrics_shard_{EVAL_SHARD_INDEX:03d}"
    return directory / f"{stem}.partial.csv", directory / f"{stem}.csv", directory / f"{stem}.json"


def run_evaluation_items(pairs, item_indices, output_partial, runner_bundle):
    rows = pd.read_csv(output_partial).to_dict("records") if output_partial.is_file() else []
    processed_items = {int(row["item"]) for row in rows}
    parity = None
    started = time.monotonic()
    for ordinal, item_index in enumerate(item_indices, 1):
        if item_index in processed_items:
            continue
        mixture, clean = next(iter(make_dataset(pairs.iloc[item_index:item_index + 1], training=False, batch_size=1)))
        noisy, target = mixture.numpy()[0], clean.numpy()[0]
        mark = time.monotonic(); fp32, fp_trace = run_fp32_stream(noisy); fp32_s = time.monotonic() - mark
        mark = time.monotonic(); int8, int8_trace = run_tflite_stream(runner_bundle, noisy); int8_s = time.monotonic() - mark
        mark = time.monotonic(); fp_metrics, int8_metrics = quality_metrics(target, fp32), quality_metrics(target, int8); metrics_s = time.monotonic() - mark
        rows.extend([{"item": item_index, "precision": "fp32", **fp_metrics}, {"item": item_index, "precision": "int8", **int8_metrics}])
        if parity is None:
            parity = {"gain_max_abs": float(np.max(np.abs(fp_trace[0][0] - int8_trace[0][0]))),
                      "coeff_max_abs": float(np.max(np.abs(fp_trace[0][1] - int8_trace[0][1]))),
                      "alpha_abs": float(abs(fp_trace[0][2] - int8_trace[0][2])),
                      "audio_max_abs": float(np.max(np.abs(fp32[:min(len(fp32), len(int8))] - int8[:min(len(fp32), len(int8))]))),
                      "host_status": "fp32_vs_int8_reported", "board_status": "board_mve_parity_pending"}
        if ordinal % EVAL_PARTIAL_EVERY == 0:
            pd.DataFrame(rows).to_csv(output_partial, index=False)
        if ordinal == 1 or ordinal % 8 == 0 or ordinal == len(item_indices):
            elapsed = time.monotonic() - started
            eta = elapsed / ordinal * (len(item_indices) - ordinal)
            print({"evaluation_item": item_index, "done": ordinal, "total": len(item_indices),
                   "fp32_s": round(fp32_s, 2), "int8_s": round(int8_s, 2), "metrics_s": round(metrics_s, 2),
                   "elapsed_s": round(elapsed, 1), "eta_s": round(eta, 1)})
    return pd.DataFrame(rows), parity


def evaluate_smoke_or_shard():
    if PRE_VELA_PATH is None:
        raise RuntimeError("Smoke/shard cần pre-Vela INT8 model")
    pairs = val_pairs if EVALUATION_MODE == "smoke" else test_pairs
    item_indices = list(range(RUN_CONFIG["metric_items"])) if EVALUATION_MODE == "smoke" else list(range(EVAL_SHARD_INDEX * EVAL_SHARD_SIZE, min((EVAL_SHARD_INDEX + 1) * EVAL_SHARD_SIZE, len(test_pairs))))
    if not item_indices:
        raise ValueError("Shard index vượt ngoài frozen test split")
    sample, _ = next(iter(make_dataset(pairs.iloc[item_indices[0]:item_indices[0] + 1], training=False, batch_size=1)))
    assert_sequence_core_parity(sample.numpy()[0])
    evaluation_started = time.monotonic()
    runner_bundle = create_tflite_runner(PRE_VELA_PATH)
    partial, completed, metadata = shard_paths()
    frame, parity = run_evaluation_items(pairs, item_indices, partial, runner_bundle)
    if EVALUATION_MODE == "smoke":
        write_smoke_package(frame, parity, evaluation_started)
        return
    expected = set(item_indices)
    if set(frame["item"].astype(int)) != expected or frame.groupby("item").size().ne(2).any():
        raise RuntimeError("Shard partial không đủ đúng hai precision cho mỗi item")
    frame.sort_values(["item", "precision"]).to_csv(completed, index=False)
    if partial.exists():
        partial.unlink()
    payload = {"schema_version": 1, "evaluation_mode": "shard", "shard": EVAL_SHARD_INDEX,
               "start": min(item_indices), "stop": max(item_indices) + 1, "items": len(item_indices),
               **evaluation_identity(), "metrics": completed.name, "parity": parity}
    metadata.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print({"shard_complete": payload})


def merge_final_shards():
    manifests = sorted(EVAL_SHARD_INPUT_ROOT.rglob("metrics_shard_*.json"))
    if len(manifests) != math.ceil(len(test_pairs) / EVAL_SHARD_SIZE):
        raise FileNotFoundError(f"Cần đủ 15 shard manifest, nhận {len(manifests)} từ {EVAL_SHARD_INPUT_ROOT}")
    payloads = [json.loads(path.read_text(encoding="utf-8")) for path in manifests]
    identities = {(item["model_sha256"], item["checkpoint_manifest_sha256"]) for item in payloads}
    if len(identities) != 1:
        raise RuntimeError("Shard dùng khác model hoặc checkpoint frozen")
    csvs = [path.with_name(item["metrics"]) for path, item in zip(manifests, payloads)]
    frame = pd.concat([pd.read_csv(path) for path in csvs], ignore_index=True)
    expected = set(range(len(test_pairs)))
    if set(frame["item"].astype(int)) != expected or frame.groupby("item").size().ne(2).any():
        raise RuntimeError("Shard overlap, thiếu item hoặc thiếu precision")
    if not FROZEN_PACKAGE_ROOT.is_dir():
        raise FileNotFoundError("Merge cần attach integration_package frozen vào FROZEN_PACKAGE_ROOT")
    shutil.copytree(FROZEN_PACKAGE_ROOT, PACKAGE, dirs_exist_ok=True)
    frame.sort_values(["item", "precision"]).to_csv(PACKAGE / "metrics_per_sample.csv", index=False)
    quality = quality_summary(frame, enforced=True)
    (PACKAGE / "metrics_summary.json").write_text(json.dumps(quality, indent=2), encoding="utf-8")
    parity = payloads[0]["parity"]
    (PACKAGE / "parity_report.json").write_text(json.dumps(parity, indent=2), encoding="utf-8")
    audit = {"shards": len(payloads), "items": len(expected), "identity": list(identities)[0],
             "quality_passed": quality["passed"], "board_status": "board_mve_parity_pending"}
    (PACKAGE / "package_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    manifest = {"schema_version": 2, "model": MODEL_NAME, "profile": "merge", "quality": quality,
                "parity": parity, "release_status": "quality_gate_passed_pending_board" if quality["passed"] else "quality_gate_failed"}
    (PACKAGE / "integration_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    if not quality["passed"]:
        raise RuntimeError(f"INT8 quality gate fail: {quality}")
    print({"merge_complete": manifest, "audit": audit})


if EVALUATION_MODE == "merge":
    merge_final_shards()
elif PRE_VELA_PATH:
    evaluate_smoke_or_shard()


{'sequence_frame_core_parity': 'passed', 'frames': 10}
{'evaluation_item': 0, 'done': 1, 'total': 512, 'fp32_s': 19.08, 'int8_s': 0.27, 'metrics_s': 0.35, 'elapsed_s': 19.8, 'eta_s': 10117.1}
{'evaluation_item': 7, 'done': 8, 'total': 512, 'fp32_s': 19.75, 'int8_s': 0.27, 'metrics_s': 0.33, 'elapsed_s': 159.8, 'eta_s': 10066.9}
{'evaluation_item': 15, 'done': 16, 'total': 512, 'fp32_s': 19.54, 'int8_s': 0.28, 'metrics_s': 0.32, 'elapsed_s': 322.7, 'eta_s': 10003.1}
{'evaluation_item': 23, 'done': 24, 'total': 512, 'fp32_s': 18.89, 'int8_s': 0.26, 'metrics_s': 0.29, 'elapsed_s': 481.3, 'eta_s': 9785.5}
{'evaluation_item': 31, 'done': 32, 'total': 512, 'fp32_s': 18.89, 'int8_s': 0.26, 'metrics_s': 0.32, 'elapsed_s': 637.7, 'eta_s': 9566.1}
{'evaluation_item': 39, 'done': 40, 'total': 512, 'fp32_s': 19.21, 'int8_s': 0.27, 'metrics_s': 0.32, 'elapsed_s': 797.0, 'eta_s': 9404.1}
{'evaluation_item': 47, 'done': 48, 'total': 512, 'fp32_s': 18.98, 'int8_s': 0.27, 'metrics_s': 0.34, 'elapsed_s'